# PC and DME Cubic Simulation

In [ ]:
# grabbing geometries that are already validated instead of building them by hand
%%bash
wget -q https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/propylene%20carbonate/SDF?record_type=3d -O pc.sdf
wget -q https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/dimethoxyethane/SDF?record_type=3d -O dme.sdf

In [ ]:
%%bash
apt-get -qq install openbabel
obabel pc.sdf -O pc.pdb
obabel dme.sdf -O dme.pdb

Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previously unselected package openbabel.
Preparing to unpack .../openbabel_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking openbabel (3.1.1+dfsg-6ubuntu5) ...
Setting up libboost-iostreams1.74.0:amd64 (

1 molecule converted
1 molecule converted


In [ ]:
def show(pdbfile):
    """Quick 3D viewer for a pdb file, just for eyeballing structures in the notebook."""
    with open(pdbfile) as f:
        pdb = f.read()
    v = py3Dmol.view(width=300, height=300)
    v.addModel(pdb, "pdb")
    v.setStyle({"stick": {}})
    v.zoomTo()
    v.show()

show("pc.pdb")
show("dme.pdb")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# packmol input for the PC/DME box
%%bash
cat > input.inp << 'EOF'
tolerance 2.0
filetype pdb
output pc_dme_box_30A.pdb

structure pc.pdb
  number 102
  inside box 0. 0. 0. 30. 30. 30.
end structure

structure dme.pdb
  number 102
  inside box 0. 0. 0. 30. 30. 30.
end structure
EOF

In [ ]:
# fire off packmol
!packmol < input.inp


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.1 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:      1234567
  Output file: pc_dme_box_30A.pdb
  Reading coordinate file: pc.pdb
  Reading coordinate file: dme.pdb
  Number of independent structures:            2
  The structures are: 
  Structure            1 :pc.pdb(          13  atoms)
  Structure            2 :dme.pdb(          16  atoms)
  Maximum number of GENCAN loops for all molecule packing:          400
  Distance tolerance:    2.000000

# DME + NaFP6

 First we take the density of the solvent then we have the volume then we have to find the number of molecules we need to fill the box to reach that density.

$$V_{\text{box}} = (30 \times 10^{-8} \text{ cm})^3 = 2.7 \times 10^{-23} \text{ cm}^3$$

$$N = \frac{\rho \cdot V_{\text{box}} \cdot N_A}{M}$$

$$\text{For DME:}$$

$$\rho = 0.867 \text{ g/cm}^3$$

$$MW = 90.12 \text{ g/mol}$$

$$N_A = 6.022 \times 10^{23} \text{ mol}^{-1}$$

$$N_{\text{DME}} = \frac{0.867 \cdot 2.7 \times 10^{-23} \cdot 6.022 \times 10^{23}}{90.12}$$

$$N_{\text{DME}} \approx 156 \text{ molecules}$$

$$V_{\text{box}} = (30 \times 10^{-8} \text{ cm})^3 = 2.7 \times 10^{-23} \text{ cm}^3$$

$$n_{\text{ions}} = \text{Molarity} \times V_{\text{liters}}$$

$$n_{\text{ions}} = 1.0 \frac{\text{mol}}{\text{L}} \times (2.7 \times 10^{-26} \text{ L}) = 2.7 \times 10^{-26} \text{ mol}$$

$$N_{\text{ion pairs}} = n_{\text{ions}} \times N_A$$

$$N_{\text{ion pairs}} = 2.7 \times 10^{-26} \text{ mol} \times 6.022 \times 10^{23} \frac{\text{molecules}}{\text{mol}}$$

$$N_{\text{ion pairs}} \approx 16.26 \approx 17 \text{ pairs}$$

$$\text{For a 60 Å box (8x Volume):}$$

$$N_{\text{ion pairs}} = 16.26 \times 8 \approx 130 \text{ pairs}$$

$$\text{Solvent Displacement Calculation for 30 Å:}$$

$$N_{\text{DME}} = N_{\text{pure}} - (N_{\text{ion pairs}} \times \frac{V_{\text{salt}}}{V_{\text{DME}}})$$

$$N_{\text{DME}} \approx 156 - 36 \approx 120 \text{ molecules}$$

1.) define the Simulation box

2.) Calculate Pure Solvent Capacity

3.) Determine Salt Ion Requirements (1 Molar)

4.) Account for Solvent Displacement

5.) Finalize the Mixture Composition

In [ ]:
%%bash
wget -q "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/dimethoxyethane/SDF?record_type=3d" -O dme.sdf

In [ ]:
def process_molecules():
    """Builds pc.pdb, dme.pdb, pf6.pdb (from SDF/SMILES), and na.pdb (hardcoded, since it's just one atom) so we've got real geometries to pack."""
    for name in ['pc', 'dme']:
        suppl = Chem.SDMolSupplier(f'{name}.sdf')
        if suppl[0]:
            mol = Chem.AddHs(suppl[0], addCoords=True)
            Chem.MolToPDBFile(mol, f'{name}.pdb')
            print(f"Generated {name}.pdb")
    pf6_smiles = "F[P-](F)(F)(F)(F)F"
    pf6 = Chem.MolFromSmiles(pf6_smiles)
    pf6 = Chem.AddHs(pf6)
    AllChem.EmbedMolecule(pf6, AllChem.ETKDG())
    Chem.MolToPDBFile(pf6, "pf6.pdb")
    print("Generated pf6.pdb")
    with open("na.pdb", "w") as f:
        f.write("HETATM    1 NA    NA A   1       0.000   0.000   0.000  1.00  0.00          Na\nEND")
    print("Generated na.pdb")
process_molecules()

Generated pc.pdb
Generated dme.pdb
Generated pf6.pdb
Generated na.pdb


[20:41:57] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[20:41:57] UFFTYPER: Unrecognized charge state for atom: 1


In [ ]:
def setup_molecules():
    """Pulls down DME's geometry and builds PF6- and Na+ pdbs so run_packmol_task has something to pack."""
    # grab DME's 3D geometry from pubchem
    os.system('wget -q "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/dimethoxyethane/SDF?record_type=3d" -O dme.sdf')

    # convert to pdb
    dme = Chem.SDMolSupplier('dme.sdf')[0]
    dme = Chem.AddHs(dme, addCoords=True)
    Chem.MolToPDBFile(dme, "dme.pdb")

    # build PF6- from SMILES since pubchem's version wasn't cooperating
    pf6 = Chem.MolFromSmiles("F[P-](F)(F)(F)(F)F")
    pf6 = Chem.AddHs(pf6)
    AllChem.EmbedMolecule(pf6, AllChem.ETKDG())
    Chem.MolToPDBFile(pf6, "pf6.pdb")
    with open("na.pdb", "w") as f:
        f.write("HETATM    1 NA    NA A   1       0.000   0.000   0.000  1.00  0.00          Na\nEND")
    print("Coordinates ready: dme.pdb, pf6.pdb, na.pdb")
def run_packmol_task():
    """Writes the packmol input for the Na/PF6/DME box and runs it."""
    input_content = """
tolerance 2.0
filetype pdb
output electrolyte_30A.pdb

structure na.pdb
  number 17
  inside cube 0. 0. 0. 30.
end structure

structure pf6.pdb
  number 17
  inside cube 0. 0. 0. 30.
end structure

structure dme.pdb
  number 120
  inside cube 0. 0. 0. 30.
end structure
"""
    with open("mixture.inp", "w") as f:
        f.write(input_content)
    os.system("packmol < mixture.inp")

setup_molecules()
run_packmol_task()

[20:43:31] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[20:43:31] UFFTYPER: Unrecognized charge state for atom: 1


Coordinates ready: dme.pdb, pf6.pdb, na.pdb


In [ ]:
view = py3Dmol.view(width=800, height=600)
with open("electrolyte_30A.pdb", "r") as f:
    pdb_data = f.read()

view.addModel(pdb_data, "pdb")
view.setStyle({'resn': 'NA'}, {'sphere': {'color': 'purple', 'scale': 0.5}})
view.setStyle({'resn': 'UNL'}, {'stick': {'radius': 0.15}})  # DME and PF6 both land in residue UNL by default
view.addStyle({'elem': 'P'}, {'sphere': {'color': 'orange', 'scale': 0.4}})
view.addStyle({'elem': 'F'}, {'sphere': {'color': 'lightgreen', 'scale': 0.3}})

view.addUnitCell()
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

If we used water instead of DME:



$$\text{For Water }(H_2O):$$

$$V_{\text{box}} = (30 \times 10^{-8} \text{ cm})^3 = 2.7 \times 10^{-23} \text{ cm}^3$$

$$\text{Pure Solvent Capacity:}$$

$$N_{\text{pure}} = \frac{\rho \cdot V_{\text{box}} \cdot N_A}{MW} = \frac{1.00 \cdot 2.7 \times 10^{-23} \cdot 6.022 \times 10^{23}}{18.015} \approx 903 \text{ molecules}$$

$$\text{Ion Pairs for 1.0 M:}$$

$$N_{\text{ion pairs}} = \text{Molarity} \times V_{\text{liters}} \times N_A \approx 17 \text{ pairs}$$

$$\text{Direct Displacement Calculation:}$$

$$V_{\text{H2O molecule}} \approx 29.9 \text{ Å}^3$$

$$V_{\text{ion pair (effective)}} \approx 360 \text{ Å}^3$$

$$N_{\text{displaced}} = \frac{N_{\text{ion-pairs}} \times V_{\text{ion-pair (effective)}}}{V_{\text{H2O molecule}}}$$

$$N_{\text{displaced}} = \frac{17 \times 360 \text{ Å}^3}{29.9 \text{ Å}^3} \approx 205 \text{ molecules}$$

$$\text{Final Count for Water Cell:}$$

$$N_{\text{H2O}} = 903 - 205 = 698 \text{ molecules}$$

So the number of NaFP6 Ion pairs remain the same, but we'd simply need to pack  more water molecules to maintain the correct density.

# Cells under here are current as of March 25th

In [ ]:
!pip install packmol
!pip install py3Dmol
!pip install rdkit
!pip install fairchem-core
!pip install ase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 56.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 M

In [ ]:
import packmol as mol
import numpy as np
import pandas as pd
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import math
import torch
from ase import units
from ase.io import Trajectory, read
from ase.md.langevin import Langevin
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from google.colab import files

In [ ]:
# these are already sitting in the colab file browser, no need to fetch them
na_file = "Sodium_Ion.pdb"
pf6_file = "hexafluorophosphate.pdb"
dme_file = "1,2-dimethoxyethane.pdb"
pc_file = "propylene_carbonate.pdb"

In [ ]:
# fairchem's uma checkpoints are gated, so this has to happen before loading the model
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# 30 Å box with DME and $NaPF_6$


In [ ]:
# 17 Na+ and 17 PF6- keeps us at 1.0 M in this 30 Å box
packmol_input = f"""
tolerance 2.0
filetype pdb
output electrolyte_30A.pdb

structure {na_file}
  number 17
  inside cube 0. 0. 0. 30.
end structure

structure {pf6_file}
  number 17
  inside cube 0. 0. 0. 30.
end structure

structure {dme_file}
  number 120
  inside cube 0. 0. 0. 30.
end structure
"""

with open("mixture.inp", "w") as f:
    f.write(packmol_input)

print("mixture.inp created. Using separate Sodium and PF6 files for randomized packing.")

mixture.inp created. Using separate Sodium and PF6 files for randomized packing.


In [ ]:
!packmol < mixture.inp


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.1 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:      1234567
  Output file: electrolyte_30A.pdb
  Reading coordinate file: Sodium_Ion.pdb
  Reading coordinate file: hexafluorophosphate.pdb
  Reading coordinate file: 1,2-dimethoxyethane.pdb
  Number of independent structures:            3
  The structures are: 
  Structure            1 :Sodium_Ion.pdb(           1  atoms)
  Structure            2 :hexafluorophosphate.pdb(           7  atoms)
  Str

In [ ]:
view = py3Dmol.view(width=800, height=600)
with open("electrolyte_30A.pdb", "r") as f:
    pdb_data = f.read()

view.addModel(pdb_data, "pdb")
# purple sodium, orange phosphorus, easier to spot in the render
view.setStyle({'resn': 'NA'}, {'sphere': {'color': 'purple', 'scale': 0.6}})
view.addStyle({'elem': 'P'}, {'sphere': {'color': 'orange', 'scale': 0.5}})
view.addStyle({'elem': 'F'}, {'sphere': {'color': 'lightgreen', 'scale': 0.3}})
view.addStyle({'resn': 'UNL'}, {'stick': {'radius': 0.08, 'opacity': 0.8}})

view.addUnitCell()
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
def calculate_counts_exact(box_size_angstrom, molarity, rho_solvent, mw_solvent):
    """Works out how many solvent and ion-pair molecules fit in the box at a given molarity, accounting for the volume the salt itself displaces."""
    N_A = 6.02214076e23

    volume_cm3 = (box_size_angstrom * 1e-8) ** 3
    volume_liters = volume_cm3 * 1e-3

    n_ion_pairs_exact = molarity * volume_liters * N_A
    # rounding up here, undershooting the concentration is worse than overshooting
    n_ion_pairs = int(math.ceil(n_ion_pairs_exact))

    n_solvent_pure = (rho_solvent * volume_cm3 / mw_solvent) * N_A
    # 2.11 is an empirical displacement factor per ion pair, back-calculated from a manual 156 to 120 solvent count
    displacement = n_ion_pairs * 2.11
    n_solvent_final = int(math.floor(n_solvent_pure - displacement))

    return n_solvent_final, n_ion_pairs

dme_count, salt_count = calculate_counts_exact(30.0, 1.0, 0.867, 90.12)
print(f"Manual Match Result:")
print(f"Sodium Ions: {salt_count}")
print(f"PF6 Ions: {salt_count}")
print(f"DME Molecules: {dme_count}")

Manual Match Result:
Sodium Ions: 17
PF6 Ions: 17
DME Molecules: 120


# 30 Å box with 1,2-dimethoxyethane and propylene_carbonate


In [ ]:
# 50:50 DME/PC, no ions, just testing the solvent mixture itself
packmol_input = f"""
tolerance 2.0
filetype pdb
output pure_binary_30A.pdb

# Volume Fraction DME
structure {dme_file}
  number 78
  inside cube 0. 0. 0. 30.
end structure

# 50% Volume Fraction PC
structure {pc_file}
  number 96
  inside cube 0. 0. 0. 30.
end structure
"""

with open("mixture.inp", "w") as f:
    f.write(packmol_input)

print("mixture.inp created for pure 50:50 DME/PC (No ions).")

mixture.inp created for pure 50:50 DME/PC (No ions).


In [ ]:
!packmol < mixture.inp


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.1 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:      1234567
  Output file: pure_binary_30A.pdb
  Reading coordinate file: 1,2-dimethoxyethane.pdb
  Reading coordinate file: propylene_carbonate.pdb
  Number of independent structures:            2
  The structures are: 
  Structure            1 :1,2-dimethoxyethane.pdb(          13  atoms)
  Structure            2 :propylene_carbonate.pdb(          13  atoms)
  Maximum number of GENCAN loops for a

In [ ]:
view = py3Dmol.view(width=800, height=600)
with open("pure_binary_30A.pdb", "r") as f:
    pdb_data = f.read()

view.addModel(pdb_data, "pdb")

view.setStyle({'resn': 'UNL'}, {'stick': {'radius': 0.15, 'colorscheme': 'Jmol'}})

# translucent surface just to get a feel for how packed it is
view.addSurface(py3Dmol.VDW, {'opacity': 0.3, 'color': 'white'}, {'resn': 'UNL'})

view.addUnitCell()
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
vol_box_cm3 = (30 * 1e-8)**3
n_dme = 78
mw_dme = 90.12
n_pc = 96
mw_pc = 102.09
n_a = 6.022e23

mass_total = ((n_dme * mw_dme) + (n_pc * mw_pc)) / n_a
density = mass_total / vol_box_cm3

print(f"Calculated Density: {density:.3f} g/cm³")
print(f"Target Density (Approx): ~1.03 g/cm³")

Calculated Density: 1.035 g/cm³
Target Density (Approx): ~1.03 g/cm³


In [ ]:
files.download('pure_binary_30A.pdb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# FairChem simulations

In [ ]:
# same gated-checkpoint login as before
from huggingface_hub import login
login()

In [ ]:
import numpy as np
from ase import units
from ase.io import Trajectory, read
from ase.md.langevin import Langevin
from fairchem.core import pretrained_mlip, FAIRChemCalculator

# using the relaxed box so the run doesn't start off with a bad-contact blowup
atoms = read("pure_binary_30A.pdb")
# UMA-1.2 via fairchem
seed = np.random.randint(0, np.iinfo(np.int32).max, dtype=int)
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device="cuda")
calc = FAIRChemCalculator(predictor, task_name="omol")
atoms.calc = calc

dyn = Langevin(
    atoms,
    timestep=0.5 * units.fs,  # good balance of speed and accuracy
    temperature_K=298.15,     # standard room temp for electrolyte work
    friction=0.01 / units.fs,
)

traj_file = "dme_pc_md.traj"
trajectory = Trajectory(traj_file, "w", atoms)
dyn.attach(trajectory.write, interval=10)  # every 10 steps, keeps the file size sane

print(f"Starting MD for {len(atoms)} atoms")
dyn.run(steps=2000)
print(f"Simulation complete! Trajectory saved to {traj_file}")

FileNotFoundError: [Errno 2] No such file or directory: 'pure_binary_30A.pdb'

### Periodic Boundary Conditions In Fairchem

In [ ]:
import numpy as np
import torch
import random
from ase import units
from ase.io import Trajectory, read
from ase.md.langevin import Langevin
from ase.optimize import LBFGS
from fairchem.core import pretrained_mlip, FAIRChemCalculator

# pinning the seeds so this run is reproducible
seed_value = 42
np.random.seed(seed_value)
torch.manual_seed(seed_value)
random.seed(seed_value)

# loading the 30 Å box and turning on PBC, otherwise this looks like a droplet instead of bulk liquid
atoms = read("pure_binary_30A.pdb")
atoms.set_cell([30.0, 30.0, 30.0])
atoms.set_pbc(True)

# UMA-1.2, GPU if we've got one
device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device=device)
calc = FAIRChemCalculator(predictor, task_name="omol")
atoms.calc = calc

# relax first to clear out any overlaps packmol left behind, otherwise the dynamics blow up immediately
print(f"Starting relaxation on {device}")
relax = LBFGS(atoms)
relax.run(fmax=0.05, steps=100)

# 0.5 fs timestep, small enough to resolve the fast vibrations
dyn = Langevin(
    atoms,
    timestep=0.5 * units.fs,
    temperature_K=298.15,
    friction=0.01 / units.fs,
)

# save a frame every 10 steps so there's something to look at later
trajectory = Trajectory("dme_pc_production.traj", "w", atoms)
dyn.attach(trajectory.write, interval=10)

print("Starting production MD for 2000 steps")
dyn.run(steps=2000)

# wrap back into the box so the final structure isn't scattered outside it
atoms.wrap()
atoms.write("final_pure_binary_30A.pdb")
print("Process finished. Your trajectory and final coordinates are ready in the file browser.")

GatedRepoError: 403 Client Error. (Request ID: Root=1-69c622ac-105ee88b661cbfd543f198ff;37a6a802-60c1-4d1d-b978-b328fb409237)

Cannot access gated repo for url https://huggingface.co/facebook/UMA/resolve/main/checkpoints/uma-s-1p2.pt.
Access to model facebook/UMA is restricted and you are not in the authorized list. Visit https://huggingface.co/facebook/UMA to ask for access.

# 04/15/2026 Stuff

## Take the d and put it into run_natfsi_npt.py

In [ ]:
import sys
from ase.io import read, Trajectory
from ase.md.npt import NPT
from ase import units
from fairchem.core import pretrained_mlip, FAIRChemCalculator
import torch, os
from copy import deepcopy
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

# Usage: python run_natfsi_npt.py <system> <temperature> [run_number]
#   system: pc, dg, tegdme
#   temperature: 253, 300, 353, etc.
#   run_number: 1 for a fresh start from xyz, 2+ to restart from a previous run
system = sys.argv[1]
temperature = int(sys.argv[2])
run = int(sys.argv[3]) if len(sys.argv) > 3 else 1

if run == 1:
    input_file = f"natfsi_{system}.xyz"  # fresh start from classical MD xyz
    fresh_start = True
else:
    prev_run = run - 1
    input_file = f"md_natfsi_{system}_T{temperature}_run{prev_run}.traj"  # pick up from the last run
    fresh_start = False

output_traj = f"md_natfsi_{system}_T{temperature}_run{run}.traj"
output_log  = f"md_natfsi_{system}_T{temperature}_run{run}.log"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Input not found: {input_file}")

base_structure = read(input_file, index=-1)
base_structure.set_pbc([True, True, True])
structure = deepcopy(base_structure)

if fresh_start:
    MaxwellBoltzmannDistribution(structure, temperature_K=temperature)
    print(f"Fresh start from {input_file}, velocities initialized at {temperature} K")
else:
    print(f"Restart from {input_file} (last frame), keeping existing velocities")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(28)
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2")
structure.calc = FAIRChemCalculator(predictor, task_name="omol")

dyn = NPT(
    atoms=structure,
    timestep=1 * units.fs,
    temperature_K=temperature,
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,
    mask=([[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
)

traj = Trajectory(output_traj, "w", structure)
dyn.attach(traj.write, interval=10)

log_fh = open(output_log, "w", buffering=1)

def print_status(a=structure, fh=log_fh):
    """Writes one status line to stdout and the log file."""
    epot = a.get_potential_energy()
    ekin = a.get_kinetic_energy()
    temp = a.get_temperature()
    vol = a.get_volume()
    cell = a.get_cell().lengths()
    line = (f"Step {dyn.nsteps:>8} | T={temp:6.1f} K | Epot={epot:10.3f} eV | "
            f"Ekin={ekin:10.3f} eV | Vol={vol:10.3f} Å³ | "
            f"Cell={cell[0]:.3f} {cell[1]:.3f} {cell[2]:.3f} Å")
    print(line); print(line, file=fh)

dyn.attach(print_status, interval=10)

print(f"{'='*60}")
print(f"System: NaTFSI/{system.upper()} | T={temperature} K | Run {run}")
print(f"Atoms: {len(structure)} | Input: {input_file}")
print(f"Initial cell: {structure.get_cell().lengths()}")
print(f"Output: {output_traj}")
print(f"{'='*60}")

dyn.run(steps=1000000)

log_fh.close()

this should create annealing_npt.xyz file. Then we put this equilibrated and wrapped extended xyz file into temp_uma_npt_parinello_rahman_new.py

In [ ]:
import sys
from ase.io import read, Trajectory
from ase.md.npt import NPT
from ase import units
from fairchem.core import pretrained_mlip, FAIRChemCalculator
import torch, os
from copy import deepcopy
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

# temperature comes in as a CLI arg
temperature = int(sys.argv[1])

#input_traj = f"md_omol_T{temperature}_re3.traj"
#input_traj = f"sys_{temperature}_Na.xyz"
input_traj =  f"md_omol_T{temperature}_run6.traj"
output_traj = f"md_omol_T{temperature}_run1_s1p2.traj"
output_log =  f"md_omol_T{temperature}_run1_s1p2.log"

if not os.path.exists(input_traj):
    raise FileNotFoundError(f"Trajectory not found: {input_traj}")

base_structure = read(input_traj, index=-1)
base_structure.set_pbc([True, True, True])
#base_structure.wrap()
structure = deepcopy(base_structure)
#MaxwellBoltzmannDistribution(structure, temperature_K=temperature)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(28)
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2")
structure.calc = FAIRChemCalculator(predictor, task_name="omol")

dyn = NPT(    atoms=structure,
    timestep = 1 * units.fs,
    temperature_K=temperature,
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,  # bigger value means a slower barostat relaxation, typical range is ~1e-3
    mask=([[1,0,0],[0,1,0],[0,0,1]]),
)

traj = Trajectory(output_traj, "w", structure)
dyn.attach(traj.write, interval=10)

log_fh = open(output_log, "w", buffering=1)

def print_status(a=structure, fh=log_fh):
    """Writes one status line to stdout and the log file."""
    epot = a.get_potential_energy()
    ekin = a.get_kinetic_energy()
    temp = a.get_temperature()
    vol = a.get_volume()
    line = (f"Step {dyn.nsteps:>8} | T={temp:6.1f} K | Epot={epot:10.3f} eV | "
            f"Ekin={ekin:10.3f} eV | Vol={vol:10.3f} Å³")
    print(line); print(line, file=fh)

dyn.attach(print_status, interval=10)
dyn.run(steps=1000000)
log_fh.close()

## To make it work in google colab, I took run_natfsi_npt.py and made a couple changes:


In [ ]:
import torch, os, sys
from ase.io import read, write, Trajectory
from ase.md.npt import NPT
from ase import units
from ase.optimize import BFGS  # BFGS over FIRE here, more stable for this system
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from fairchem.core import pretrained_mlip, FAIRChemCalculator

input_pdb = "napf6_dme_30_1m.pdb"
atoms = read(input_pdb)
atoms.set_cell([30.0, 30.0, 30.0])
atoms.set_pbc(True)
atoms.wrap()

# forcing GPU, this thing's too slow on CPU to bother
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# loading the model and turning off checkpointing
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device=str(device))
if hasattr(predictor, 'model'):
    predictor.model.use_checkpoint = False  # avoids a CheckpointError we kept hitting otherwise

atoms.calc = FAIRChemCalculator(predictor, task_name="omol")

# quick relax to shake out bad contacts from packing
print("Starting relaxation to fix bad contacts...")
relax = BFGS(atoms, logfile=None)
relax.run(fmax=5.0, steps=10)
print("Relaxation complete.")

temperature = 298
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature)

dyn = NPT(
    atoms=atoms,
    timestep=1.0 * units.fs,
    temperature_K=temperature,
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,
    mask=([[1,0,0],[0,1,0],[0,0,1]]),
)

output_extxyz = "napf6_equilibrated_final.xyz"
def print_status():
    """Prints where the run's at: step, temp, volume."""
    temp = atoms.get_temperature()
    vol = atoms.get_volume()
    print(f"Step {dyn.nsteps:4d} | Temp: {temp:6.1f}K | Volume: {vol:10.2f} A^3")

dyn.attach(print_status, interval=5)

print("Starting production NPT")
dyn.run(steps=500)  # short run first, just to make sure nothing explodes

write(output_extxyz, atoms, format="extxyz")
print(f"Success! Saved to {output_extxyz}")

/usr/local/lib/python3.12/dist-packages/ase/io/proteindatabank.py:112: UserWarning: Length of occupancy array, 17, different from number of atoms 2136
  warnings.warn('Length of {} array, {}, '


Using device: cpu
Starting relaxation to fix bad contacts...


KeyboardInterrupt: 

## now, the outputted file should be similar to annealing_npt.xyz so we run the google colab friendly version of temp_uma_npt_parinello_rahman_new.py with both of the files to then compare

In [ ]:
# picks up the napf6_equilibrated_final.xyz output from the last cell
import torch, os
from ase.io import read, Trajectory
from ase.md.npt import NPT
from ase import units
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from copy import deepcopy

temperature = 298  # bump this if you want a different run temp
input_file = "napf6_equilibrated_final.xyz"
output_traj = f"production_napf6_T{temperature}.traj"
output_log  = f"production_napf6_T{temperature}.log"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"File not found: {input_file}")

structure = read(input_file, index=-1)
structure.set_pbc([True, True, True])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(28)
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2")
structure.calc = FAIRChemCalculator(predictor, task_name="omol")

# Parrinello-Rahman style NPT
dyn = NPT(
    atoms=structure,
    timestep=1 * units.fs,
    temperature_K=temperature,
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,
    mask=([[1,0,0],[0,1,0],[0,0,1]]),
)

traj = Trajectory(output_traj, "w", structure)
dyn.attach(traj.write, interval=10)
log_fh = open(output_log, "w", buffering=1)

def print_status(a=structure, fh=log_fh):
    """Logs one line to stdout and the log file."""
    epot = a.get_potential_energy(); ekin = a.get_kinetic_energy()
    temp = a.get_temperature(); vol = a.get_volume()
    line = f"Step {dyn.nsteps:>8} | T={temp:6.1f} K | Epot={epot:10.3f} eV | Vol={vol:10.3f} Å³"
    print(line); print(line, file=fh)

dyn.attach(print_status, interval=100)
dyn.run(steps=100000)  # bump this up or down depending on how long you've got
log_fh.close()

In [ ]:
import torch, os
from ase.io import read, Trajectory
from ase.md.npt import NPT
from ase import units
from fairchem.core import pretrained_mlip, FAIRChemCalculator

temperature = 298
input_file = "annealing_npt.xyz"
output_traj = "production_results.traj"
output_log  = "production_results.log"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Ensure {input_file} is uploaded to Colab.")

structure = read(input_file, index=-1)
structure.set_pbc([True, True, True])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_str = str(device)
print(f"Using device: {device_str}")

predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device=device_str)

# disabling checkpointing, fixes a CheckpointError we kept running into
if hasattr(predictor, 'model'):
    predictor.model.use_checkpoint = False
elif hasattr(predictor, 'inference_model'):
    predictor.inference_model.use_checkpoint = False

structure.calc = FAIRChemCalculator(predictor, task_name="omol")

# NPT needs some initial velocity, so kick it with MB if the file came in cold
current_temp = structure.get_temperature()
if current_temp < 1.0:
    from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
    MaxwellBoltzmannDistribution(structure, temperature_K=temperature)
    print(f"Initial Temperature was 0K. Resetting to {temperature}K.")
else:
    print(f"Continuing with existing temperature: {current_temp:.1f}K")

dyn = NPT(
    atoms=structure,
    timestep=1.0 * units.fs,
    temperature_K=temperature,
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,
    mask=([[1,0,0],[0,1,0],[0,0,1]]),
)

traj = Trajectory(output_traj, "w", structure)
dyn.attach(traj.write, interval=50)

def print_status():
    """Quick status line: temp and volume."""
    temp = structure.get_temperature()
    vol = structure.get_volume()
    print(f"Step {dyn.nsteps:>5} | Temp: {temp:6.1f} K | Volume: {vol:10.2f} A^3")

"""# We use a 10-step interval for the first check to see progress quickly
dyn.attach(print_status, interval=10)

print("Starting production run...")
dyn.run(steps=100000)"""

# logging every 5 steps so we see progress fast during this smoke test
dyn.attach(print_status, interval=5)

print("Starting 50-step Speed Test...")
dyn.run(steps=50)
print("Test complete! If you see Step 50 above, the code is working perfectly.")

Using device: cuda


/tmp/ipykernel_21836/1019261555.py:47: FutureWarning: NPT thermostat has been moved/renamed to ase.md.melchionna.MelchionnaNPT. Please use this class instead (or a newer NPT thermostat!)
  dyn = NPT(


Initial Temperature was 0K. Resetting to 298K.
Starting 50-step Speed Test...
Step     0 | Temp:  302.1 K | Volume:   23384.33 A^3
Step     5 | Temp:  342.3 K | Volume:   23374.30 A^3
Step    10 | Temp:  320.4 K | Volume:   22817.21 A^3
Step    15 | Temp:  304.8 K | Volume:   23088.26 A^3
Step    20 | Temp:  310.9 K | Volume:   22984.07 A^3
Step    25 | Temp:  278.6 K | Volume:   22771.59 A^3
Step    30 | Temp:  298.6 K | Volume:   22205.90 A^3
Step    35 | Temp:  302.9 K | Volume:   22302.38 A^3
Step    40 | Temp:  303.1 K | Volume:   22339.26 A^3
Step    45 | Temp:  305.6 K | Volume:   22221.51 A^3
Step    50 | Temp:  343.7 K | Volume:   22090.01 A^3
Test complete! If you see Step 50 above, the code is working perfectly.


In [ ]:
from ase.io import read, write

configs = read('production_results.traj', index=':')

write('production_vmd.xyz', configs)
print("Conversion complete! Download production_vmd.xyz to view in VMD.")

Conversion complete! Download production_vmd.xyz to view in VMD.
